# Solutions: Notebook 10 - Dynamic Quantile Regression

Complete solutions for all exercises from the Dynamic Quantile Regression Models notebook.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from panelbox.core.panel_data import PanelData
from panelbox.models.quantile import DynamicQuantile, PooledQuantile

warnings.filterwarnings("ignore", category=UserWarning)
np.random.seed(42)

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "font.size": 11,
    }
)

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
PLOTS_DIR = OUTPUT_DIR / "plots"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)


def make_X(data, *cols):
    """Build design matrix with intercept."""
    arrays = [np.ones(len(data))] + [data[c].values.astype(float) for c in cols]
    return np.column_stack(arrays)


# Load data
data = pd.read_csv(DATA_DIR / "firm_production.csv")
data = data.sort_values(["firm_id", "year"]).reset_index(drop=True)
data["profit_lag1"] = data.groupby("firm_id")["profit"].shift(1)
data["profit_lag2"] = data.groupby("firm_id")["profit"].shift(2)
data_clean = data.dropna(subset=["profit_lag1", "profit_lag2"]).copy()

print(f"Data loaded: {len(data_clean)} observations, {data_clean['firm_id'].nunique()} firms")
print("Setup complete.")

---

## Exercise 1: Endogeneity in Dynamic Panels (Easy)

**Task**: Explain why $Y_{it-1}$ is endogenous in dynamic panel models. Why doesn't the within-transformation fix it?

In [ ]:
print("SOLUTION: Endogeneity in Dynamic Panels")
print("=" * 70)
print()
print("WHY IS Y_{it-1} ENDOGENOUS?")
print("-" * 40)
print("""
In the dynamic panel model:

  Y_it = alpha_i + rho * Y_{it-1} + eps_it

Y_{it-1} is endogenous because:

  Y_{it-1} = alpha_i + rho * Y_{it-2} + eps_{it-1}

So Y_{it-1} depends on alpha_i through the recursion:
  Y_{it-1} = alpha_i * sum(rho^j for j=0..t-2) + ...

Therefore: Corr(Y_{it-1}, alpha_i) != 0
Since alpha_i is in the error term, Y_{it-1} is correlated
with the composite error (alpha_i + eps_it).

This is NOT the same as reverse causality -- it's the
incidental parameters problem specific to dynamic panels.
""")
print()
print("WHY DOESN'T WITHIN-TRANSFORMATION FIX IT?")
print("-" * 40)
print("""
The within transformation demeans by entity:

  (Y_it - Y_i_bar) = rho * (Y_{it-1} - Y_i_bar) + (eps_it - eps_i_bar)

The problem: Y_{it-1} is correlated with eps_i_bar because
eps_i_bar = (1/T) * sum(eps_is for s=1..T), which includes
eps_{it} and later periods.

Since Y_{it-1} = ... + eps_{it-1}, and eps_i_bar contains
eps_{it-1}, the demeaned regressor is correlated with the
demeaned error.

This creates the "Nickell bias" which is O(1/T):
  - For small T: bias is substantial
  - For large T: bias vanishes
  - The bias is UPWARD (rho is overestimated)

The solution is to use deeper lags (Y_{it-2}, Y_{it-3}, ...)
as instruments, as in Arellano-Bond or Galvao (2011) IV-QR.
""")

# Demonstrate the Nickell bias
print("NUMERICAL DEMONSTRATION:")
print("-" * 40)

y = data_clean["profit"].values
X_naive = make_X(data_clean, "profit_lag1")
entity_ids = data_clean["firm_id"].values

# Naive OLS
from numpy.linalg import lstsq

beta_ols = lstsq(X_naive, y, rcond=None)[0]
print(f"  Naive OLS rho:     {beta_ols[1]:.4f} (biased upward)")

# Naive QR at median
model_naive = PooledQuantile(y, X_naive, entity_id=entity_ids, quantiles=0.5)
result_naive = model_naive.fit(se_type="cluster")
print(f"  Naive QR(0.5) rho: {result_naive.params.ravel()[1]:.4f} (also biased)")

# IV-QR for comparison
panel = PanelData(data, entity_col="firm_id", time_col="year")
dyn_model = DynamicQuantile(panel, formula="profit ~ size", tau=0.5, lags=1, method="iv")
iv_result = dyn_model.fit(iv_lags=2, verbose=False)
print(f"  IV-QR(0.5) rho:    {iv_result.results[0.5].persistence[0]:.4f} (corrected)")
print(f"\n  Bias magnitude:    {beta_ols[1] - iv_result.results[0.5].persistence[0]:.4f}")

---

## Exercise 2: Manual QAR(1) (Easy)

**Task**: Implement a simple QAR(1) model manually and plot the persistence across fine quantile grid.

In [ ]:
# Fine quantile grid
tau_fine = [
    0.05,
    0.1,
    0.15,
    0.2,
    0.25,
    0.3,
    0.35,
    0.4,
    0.45,
    0.5,
    0.55,
    0.6,
    0.65,
    0.7,
    0.75,
    0.8,
    0.85,
    0.9,
    0.95,
]

y_ar = data_clean["profit"].values
X_ar = make_X(data_clean, "profit_lag1")
entity_ar = data_clean["firm_id"].values

rho_path = []
se_path = []

for tau in tau_fine:
    model = PooledQuantile(y_ar, X_ar, entity_id=entity_ar, quantiles=tau)
    result = model.fit(se_type="cluster")
    rho_path.append(result.params.ravel()[1])
    se_path.append(result.std_errors.ravel()[1])

rho_path = np.array(rho_path)
se_path = np.array(se_path)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(tau_fine, rho_path, "o-", linewidth=2, markersize=6, color="#0072B2", label="rho(tau)")
ax.fill_between(
    tau_fine, rho_path - 1.96 * se_path, rho_path + 1.96 * se_path, alpha=0.15, color="#0072B2"
)

# OLS reference
rho_ols = lstsq(X_ar, y_ar, rcond=None)[0][1]
ax.axhline(rho_ols, color="red", linestyle="--", linewidth=1.5, label=f"OLS rho = {rho_ols:.3f}")
ax.axhline(1, color="black", linestyle=":", alpha=0.3)
ax.axhline(0, color="black", linestyle=":", alpha=0.3)

ax.set_xlabel("Quantile (tau)", fontsize=12, fontweight="bold")
ax.set_ylabel("Persistence rho(tau)", fontsize=12, fontweight="bold")
ax.set_title("QAR(1): Persistence Coefficient Path", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("Key observations:")
print(f"  rho(0.05) = {rho_path[0]:.4f}  (low quantile: LOW persistence)")
print(f"  rho(0.50) = {rho_path[9]:.4f}  (median)")
print(f"  rho(0.95) = {rho_path[-1]:.4f}  (high quantile: HIGH persistence)")
print(f"  Range: {rho_path.max() - rho_path.min():.4f}")
print()
print("The persistence is INCREASING in tau, confirming:")
print('"Winners keep winning, losers are volatile"')

---

## Exercise 3: Instrument Validity (Medium)

**Task**: Test whether $Y_{it-2}$ is a valid instrument.

In [ ]:
print("INSTRUMENT VALIDITY TEST: Y_{it-2} as instrument for Y_{it-1}")
print("=" * 70)

# 1. RELEVANCE: Is Y_{it-2} correlated with Y_{it-1}?
print("\n1. RELEVANCE (should be strong):")
corr_relevance = np.corrcoef(data_clean["profit_lag2"].values, data_clean["profit_lag1"].values)[
    0, 1
]
print(f"   Corr(Y_{{it-2}}, Y_{{it-1}}) = {corr_relevance:.4f}")

# First-stage F-statistic
from numpy.linalg import lstsq

X_fs = make_X(data_clean, "profit_lag2", "size", "capital", "labor")
y_fs = data_clean["profit_lag1"].values
beta_fs = lstsq(X_fs, y_fs, rcond=None)[0]
y_hat_fs = X_fs @ beta_fs
ss_reg = np.sum((y_hat_fs - y_fs.mean()) ** 2)
ss_res = np.sum((y_fs - y_hat_fs) ** 2)
n_fs = len(y_fs)
k_fs = X_fs.shape[1]
f_stat = (ss_reg / (k_fs - 1)) / (ss_res / (n_fs - k_fs))
print(f"   First-stage F-statistic = {f_stat:.2f}")
print("   Rule of thumb: F > 10 is strong instrument")
print(f"   Result: {'STRONG' if f_stat > 10 else 'WEAK'} instrument")

# 2. EXCLUSION: Is Y_{it-2} uncorrelated with eps_it?
print("\n2. EXCLUSION RESTRICTION (Y_{it-2} should not directly affect Y_it):")

# Estimate the model first
y_main = data_clean["profit"].values
X_main = make_X(data_clean, "profit_lag1", "size", "capital", "labor")
beta_main = lstsq(X_main, y_main, rcond=None)[0]
residuals = y_main - X_main @ beta_main

# Regress residuals on Y_{it-2}
corr_excl = np.corrcoef(residuals, data_clean["profit_lag2"].values)[0, 1]
print(f"   Corr(residuals, Y_{{it-2}}) = {corr_excl:.4f}")

# Formal test: regress residuals on instrument
X_excl = make_X(data_clean, "profit_lag2")
beta_excl = lstsq(X_excl, residuals, rcond=None)[0]
resid_excl = residuals - X_excl @ beta_excl
n_e = len(residuals)
se_excl = np.sqrt(
    np.sum(resid_excl**2)
    / (n_e - 2)
    / np.sum((data_clean["profit_lag2"].values - data_clean["profit_lag2"].mean()) ** 2)
)
t_stat_excl = beta_excl[1] / se_excl
print(f"   t-statistic on Y_{{it-2}}: {t_stat_excl:.4f}")
print(f"   Result: {'FAILS' if abs(t_stat_excl) > 1.96 else 'PASSES'} exclusion test at 5%")

print()
print("NOTE: The exclusion restriction cannot be tested directly.")
print("The test above only checks if the instrument is orthogonal")
print("to residuals from the BIASED model. A proper test requires")
print("overidentifying restrictions (Sargan/Hansen J-test).")

---

## Exercise 4: Simulate Heterogeneous Persistence (Medium)

**Task**: Simulate AR(1) data with quantile-dependent persistence and verify recovery.

In [ ]:
from scipy.stats import norm

np.random.seed(42)

# Parameters
N_sim, T_sim = 500, 50
def rho_func(u):
    return 0.3 + 0.4 * u  # rho(0) = 0.3, rho(1) = 0.7

# Simulate
Y = np.zeros((N_sim, T_sim))
Y[:, 0] = np.random.normal(0, 1, N_sim)

for t in range(1, T_sim):
    u = np.random.uniform(0, 1, N_sim)
    rho_t = rho_func(u)
    eps = norm.ppf(u) * 0.5
    Y[:, t] = rho_t * Y[:, t - 1] + eps

# Create panel
ids = np.repeat(np.arange(N_sim), T_sim - 1)
y_sim = Y[:, 1:].ravel()
y_lag_sim = Y[:, :-1].ravel()
X_sim = np.column_stack([np.ones(len(y_sim)), y_lag_sim])
entity_sim = ids

# Estimate QR at different quantiles
tau_test = [0.1, 0.25, 0.5, 0.75, 0.9]
rho_estimated = []
rho_true = []

print("SIMULATION RECOVERY TEST")
print("=" * 60)
print(f"{'tau':<8} {'True rho':<14} {'Estimated rho':<16} {'Error':<10}")
print("-" * 60)

for tau in tau_test:
    model = PooledQuantile(y_sim, X_sim, entity_id=entity_sim, quantiles=tau)
    result = model.fit(se_type="cluster")
    rho_est = result.params.ravel()[1]
    rho_true_val = rho_func(tau)
    rho_estimated.append(rho_est)
    rho_true.append(rho_true_val)
    error = rho_est - rho_true_val
    print(f"{tau:<8.2f} {rho_true_val:<14.4f} {rho_est:<16.4f} {error:<10.4f}")

print("=" * 60)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

tau_dense = np.linspace(0.05, 0.95, 50)
ax.plot(
    tau_dense,
    [rho_func(t) for t in tau_dense],
    "r--",
    linewidth=2,
    label="True rho(tau)",
    alpha=0.7,
)
ax.plot(tau_test, rho_estimated, "bo-", linewidth=2, markersize=8, label="Estimated rho(tau)")

ax.set_xlabel("Quantile (tau)", fontsize=12, fontweight="bold")
ax.set_ylabel("Persistence rho(tau)", fontsize=12, fontweight="bold")
ax.set_title("Simulation Recovery: Heterogeneous Persistence", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print("\nConclusion: Pooled QR successfully recovers the heterogeneous")
print("persistence pattern from the simulated data.")

---

## Exercise 5: Bootstrap Bias Correction (Hard)

**Task**: Implement bootstrap-based bias correction for the persistence parameter.

In [ ]:
import time

np.random.seed(42)

B = 199
tau_bc = 0.5

# Original estimate
y_bc = data_clean["profit"].values
X_bc = make_X(data_clean, "profit_lag1", "size", "capital", "labor")
entity_bc = data_clean["firm_id"].values

model_orig = PooledQuantile(y_bc, X_bc, entity_id=entity_bc, quantiles=tau_bc)
result_orig = model_orig.fit(se_type="cluster")
rho_original = result_orig.params.ravel()[1]

print(f"Original naive rho(0.5) = {rho_original:.4f}")
print(f"Running cluster bootstrap (B={B})...")

# Cluster bootstrap
entities = data_clean["firm_id"].unique()
boot_rhos = []

t0 = time.time()
for _b in range(B):
    # Resample entities
    boot_entities = np.random.choice(entities, len(entities), replace=True)

    # Collect all observations for sampled entities
    boot_idx = []
    for ent in boot_entities:
        ent_mask = data_clean["firm_id"].values == ent
        boot_idx.extend(np.where(ent_mask)[0])
    boot_idx = np.array(boot_idx)

    y_boot = y_bc[boot_idx]
    X_boot = X_bc[boot_idx]
    eid_boot = entity_bc[boot_idx]

    try:
        model_boot = PooledQuantile(y_boot, X_boot, entity_id=eid_boot, quantiles=tau_bc)
        result_boot = model_boot.fit(se_type="nonrobust")
        boot_rhos.append(result_boot.params.ravel()[1])
    except Exception:
        pass

elapsed = time.time() - t0
boot_rhos = np.array(boot_rhos)

print(f"Completed in {elapsed:.1f}s ({len(boot_rhos)} valid replications)")

# Bias correction
boot_mean = np.mean(boot_rhos)
bias = boot_mean - rho_original
rho_corrected = rho_original - bias

print()
print("BIAS CORRECTION RESULTS")
print("=" * 50)
print(f"  Original rho:      {rho_original:.4f}")
print(f"  Bootstrap mean:    {boot_mean:.4f}")
print(f"  Estimated bias:    {bias:.4f}")
print(f"  Corrected rho:     {rho_corrected:.4f}")
print(f"  Bootstrap SE:      {np.std(boot_rhos):.4f}")
print(
    f"  95% CI (percentile): [{np.percentile(boot_rhos, 2.5):.4f}, {np.percentile(boot_rhos, 97.5):.4f}]"
)
print()
print("Note: The bootstrap bias correction is approximate.")
print("For more rigorous correction, use the DynamicQuantile")
print('class with method="iv" which uses deeper lags as instruments.')

# Visualize bootstrap distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(boot_rhos, bins=40, alpha=0.7, edgecolor="white", density=True, color="steelblue")
ax.axvline(
    rho_original, color="red", linestyle="--", linewidth=2, label=f"Original = {rho_original:.4f}"
)
ax.axvline(
    rho_corrected,
    color="green",
    linestyle="-",
    linewidth=2,
    label=f"Corrected = {rho_corrected:.4f}",
)
ax.set_xlabel("rho(0.5)", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Bootstrap Distribution of Persistence Parameter", fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---

## Exercise 6: Income Mobility Application (Hard)

**Task**: Use dynamic QR to study income persistence using wage data.

In [ ]:
# Load wage data
wage_data = pd.read_csv(DATA_DIR / "card_education.csv")
wage_data = wage_data.sort_values(["id", "year"]).reset_index(drop=True)
wage_data["lwage_lag1"] = wage_data.groupby("id")["lwage"].shift(1)
wage_data["lwage_lag2"] = wage_data.groupby("id")["lwage"].shift(2)
wage_clean = wage_data.dropna(subset=["lwage_lag1", "lwage_lag2"]).copy()

print(f"Wage panel: {len(wage_clean)} obs, {wage_clean['id'].nunique()} individuals")
print(f"Years: {wage_clean['year'].nunique()}")
print()

# Naive dynamic QR
tau_wage = [0.1, 0.25, 0.5, 0.75, 0.9]

y_wage = wage_clean["lwage"].values
X_wage = make_X(wage_clean, "lwage_lag1", "educ", "exper")
entity_wage = wage_clean["id"].values

print("INCOME PERSISTENCE (Naive Pooled QR)")
print("=" * 70)
print(f"{'tau':<8} {'rho(tau)':<12} {'SE':<12} {'Half-life':<14}")
print("-" * 70)

wage_results = {}
for tau in tau_wage:
    model = PooledQuantile(y_wage, X_wage, entity_id=entity_wage, quantiles=tau)
    result = model.fit(se_type="cluster")
    rho = result.params.ravel()[1]
    se = result.std_errors.ravel()[1]
    hl = np.log(0.5) / np.log(rho) if 0 < rho < 1 else np.nan
    hl_str = f"{hl:.1f} years" if not np.isnan(hl) else "N/A"
    wage_results[tau] = {"rho": rho, "se": se, "hl": hl}
    print(f"{tau:<8.2f} {rho:<12.4f} {se:<12.4f} {hl_str:<14}")
print("=" * 70)

In [ ]:
# IV-QR for wages using DynamicQuantile
panel_wage = PanelData(wage_data, entity_col="id", time_col="year")

try:
    dyn_wage = DynamicQuantile(
        panel_wage, formula="lwage ~ educ + exper", tau=tau_wage, lags=1, method="iv"
    )
    iv_wage_results = dyn_wage.fit(iv_lags=2, verbose=False)

    print("\nINCOME PERSISTENCE (IV-QR Corrected)")
    print("=" * 70)
    print(f"{'tau':<8} {'Naive rho':<12} {'IV rho':<12} {'Bias':<12} {'Half-life':<14}")
    print("-" * 70)
    for tau in tau_wage:
        rho_naive = wage_results[tau]["rho"]
        rho_iv = iv_wage_results.results[tau].persistence[0]
        bias = rho_naive - rho_iv
        hl = np.log(0.5) / np.log(rho_iv) if 0 < rho_iv < 1 else np.nan
        hl_str = f"{hl:.1f} yr" if not np.isnan(hl) else "N/A"
        print(f"{tau:<8.2f} {rho_naive:<12.4f} {rho_iv:<12.4f} {bias:<12.4f} {hl_str:<14}")
    print("=" * 70)

except Exception as e:
    print(f"IV-QR error: {e}")
    print("Using naive results only.")
    iv_wage_results = None

In [ ]:
# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Persistence path
rho_naive_wage = [wage_results[tau]["rho"] for tau in tau_wage]
se_naive_wage = [wage_results[tau]["se"] for tau in tau_wage]

ax1.plot(
    tau_wage,
    rho_naive_wage,
    "o-",
    linewidth=2,
    markersize=8,
    color="gray",
    alpha=0.6,
    label="Naive QR",
)
ax1.fill_between(
    tau_wage,
    [r - 1.96 * s for r, s in zip(rho_naive_wage, se_naive_wage)],
    [r + 1.96 * s for r, s in zip(rho_naive_wage, se_naive_wage)],
    alpha=0.1,
    color="gray",
)

if iv_wage_results is not None:
    rho_iv_wage = [iv_wage_results.results[tau].persistence[0] for tau in tau_wage]
    se_iv_wage = [iv_wage_results.results[tau].std_errors[0] for tau in tau_wage]
    ax1.plot(
        tau_wage, rho_iv_wage, "s-", linewidth=2.5, markersize=9, color="#0072B2", label="IV-QR"
    )
    ax1.fill_between(
        tau_wage,
        [r - 1.96 * s for r, s in zip(rho_iv_wage, se_iv_wage)],
        [r + 1.96 * s for r, s in zip(rho_iv_wage, se_iv_wage)],
        alpha=0.15,
        color="#0072B2",
    )

ax1.axhline(1, color="black", linestyle=":", alpha=0.3)
ax1.axhline(0, color="black", linestyle=":", alpha=0.3)
ax1.set_xlabel("Quantile (tau)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Wage Persistence rho(tau)", fontsize=12, fontweight="bold")
ax1.set_title("Panel A: Income Persistence", fontsize=13, fontweight="bold")
ax1.legend(fontsize=10)

# Panel B: Half-lives
hl_vals = [wage_results[tau]["hl"] for tau in tau_wage]
colors_hl = ["#2166ac" if not np.isnan(hl) else "#d6604d" for hl in hl_vals]
ax2.bar(tau_wage, hl_vals, alpha=0.7, color=colors_hl, edgecolor="black", width=0.08)
ax2.set_xlabel("Quantile (tau)", fontsize=12, fontweight="bold")
ax2.set_ylabel("Half-Life (years)", fontsize=12, fontweight="bold")
ax2.set_title("Panel B: Income Shock Half-Life", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

print("\nINTERPRETATION:")
print("=" * 60)
print("High-income individuals (tau=0.9) show higher persistence:")
print("  -> Income shocks at the top are more persistent")
print('  -> "The rich stay rich"')
print()
print("Low-income individuals (tau=0.1) show lower persistence:")
print("  -> Income is more volatile at the bottom")
print('  -> "The poor fluctuate more"')
print()
print("This has POLICY IMPLICATIONS:")
print("  - Transitory income support helps low earners (volatile income)")
print("  - Permanent programs needed for structural mobility")
print("  - Standard models miss this heterogeneity")